In [3]:
%pip install scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 114.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 151.7 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 145.4 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
Note: you may need to restart the kernel to use updated packages.


In [1]:
import sys
print("Python version:", sys.version)

import pandas as pd
import numpy as np
import sklearn

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Environment ready!")

Python version: 3.12.7 | packaged by Anaconda, Inc. | (main, Oct  4 2024, 13:27:36) [GCC 11.2.0]
Pandas: 2.2.3
NumPy: 2.5.4
Scikit-learn: 1.9.1
Environment ready!


In [3]:
%pip install ucimlrepo

Note: you may need to restart the kernel to use updated packages.


In [1]:
from ucimlrepo import fetch_ucirepo

credit_data = fetch_ucirepo(id=350)

X = credit_data.data.features.copy()
y = credit_data.data.targets.iloc[:, 0].copy()

y.name = "default"

print("Input data shape:", X.shape)
print("Target shape:", y.shape)

print("\nFeature names:")
print(X.columns.tolist())

print("\nMissing values:", X.isna().sum().sum())
print("Missing target values:", y.isna().sum())

print("\nTarget distribution:")
print(y.value_counts().sort_index())

display(X.head())

Input data shape: (30000, 23)
Target shape: (30000,)

Feature names:
['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8', 'X9', 'X10', 'X11', 'X12', 'X13', 'X14', 'X15', 'X16', 'X17', 'X18', 'X19', 'X20', 'X21', 'X22', 'X23']

Missing values: 0
Missing target values: 0

Target distribution:
default
0    23364
1     6636
Name: count, dtype: int64


,X1,X2,X3,X4,X5,X6,X7,X8,X9,X10,...,X14,X15,X16,X17,X18,X19,X20,X21,X22,X23
0,20000,2,2,1,24,2,2,-1,-1,-2,...,689,0,0,0,0,689,0,0,0,0
1,120000,2,2,2,26,-1,2,0,0,0,...,2682,3272,3455,3261,0,1000,1000,1000,0,2000
2,90000,2,2,2,34,0,0,0,0,0,...,13559,14331,14948,15549,1518,1500,1000,1000,1000,5000
3,50000,2,2,1,37,0,0,0,0,0,...,49291,28314,28959,29547,2000,2019,1200,1100,1069,1000
4,50000,1,2,1,57,-1,0,-1,0,0,...,35835,20940,19146,19131,2000,36681,10000,9000,689,679


In [3]:
from pathlib import Path
from sklearn.model_selection import train_test_split

y = y.astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("\nDefault percentage in training:")
print(y_train.value_counts(normalize=True).sort_index().mul(100).round(2))

print("\nDefault percentage in testing:")
print(y_test.value_counts(normalize=True).sort_index().mul(100).round(2))


test_data = X_test.copy()
test_data["default"] = y_test

test_data.to_csv("test_data.csv", index=False)

Path("model").mkdir(exist_ok=True)

print("\nSaved test_data.csv and created model/ folder.")

Training features: (24000, 23)
Testing features: (6000, 23)

Default percentage in training:
default
0    77.88
1    22.12
Name: proportion, dtype: float64

Default percentage in testing:
default
0    77.88
1    22.12
Name: proportion, dtype: float64

Saved test_data.csv and created model/ folder.


In [5]:
import joblib
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix
)

categorical_columns = ["X2", "X3", "X4"]
numerical_columns = [
    column for column in X_train.columns
    if column not in categorical_columns
]

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), numerical_columns),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            categorical_columns
        )
    ]
)

def evaluate_model(model, features, target):
    predictions = model.predict(features)

    if hasattr(model, "predict_proba"):
        positive_index = list(model.classes_).index(1)
        scores = model.predict_proba(features)[:, positive_index]
    else:
        scores = model.decision_function(features)

    return {
        "Accuracy": accuracy_score(target, predictions),
        "AUC": roc_auc_score(target, scores),
        "Precision": precision_score(target, predictions, zero_division=0),
        "Recall": recall_score(target, predictions, zero_division=0),
        "F1": f1_score(target, predictions, zero_division=0),
        "MCC": matthews_corrcoef(target, predictions)
    }

logistic_model = Pipeline(
    steps=[
        ("preprocessing", preprocessor),
        ("classifier", LogisticRegression(max_iter=2000, random_state=42))
    ]
)

logistic_model.fit(X_train, y_train)

results = {
    "Logistic Regression": evaluate_model(
        logistic_model, X_test, y_test
    )
}

joblib.dump(logistic_model, "model/logistic_regression.joblib")

display(pd.DataFrame.from_dict(results, orient="index").round(4))

print("Confusion matrix: rows = actual, columns = predicted; order = [0, 1]")
print(confusion_matrix(y_test, logistic_model.predict(X_test), labels=[0, 1]))

print("\nSaved model/logistic_regression.joblib")

,Accuracy,AUC,Precision,Recall,F1,MCC
Logistic Regression,0.8088,0.71,0.6923,0.2442,0.361,0.3302


Confusion matrix: rows = actual, columns = predicted; order = [0, 1]
[[4529  144]
 [1003  324]]

Saved model/logistic_regression.joblib


In [7]:
from sklearn.base import clone
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

decision_tree_model = Pipeline(
    steps=[
        ("preprocessing", clone(preprocessor)),
        (
            "classifier",
            DecisionTreeClassifier(
                max_depth=6,
                min_samples_leaf=20,
                random_state=42
            )
        )
    ]
)

knn_model = Pipeline(
    steps=[
        ("preprocessing", clone(preprocessor)),
        (
            "classifier",
            KNeighborsClassifier(
                n_neighbors=15,
                n_jobs=-1
            )
        )
    ]
)

new_models = {
    "Decision Tree": (
        decision_tree_model,
        "model/decision_tree.joblib"
    ),
    "KNN": (
        knn_model,
        "model/knn.joblib"
    )
}

for name, (model, file_path) in new_models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    results[name] = evaluate_model(model, X_test, y_test)
    joblib.dump(model, file_path)

    print(f"Saved {file_path}")
    print("Confusion matrix: rows = actual, columns = predicted; order = [0, 1]")
    print(confusion_matrix(y_test, model.predict(X_test), labels=[0, 1]))
    print()

display(pd.DataFrame.from_dict(results, orient="index").round(4))

Training Decision Tree...
Saved model/decision_tree.joblib
Confusion matrix: rows = actual, columns = predicted; order = [0, 1]
[[4449  224]
 [ 890  437]]

Training KNN...
Saved model/knn.joblib
Confusion matrix: rows = actual, columns = predicted; order = [0, 1]
[[4398  275]
 [ 873  454]]



,Accuracy,AUC,Precision,Recall,F1,MCC
Logistic Regression,0.8088,0.7100,0.6923,0.2442,0.3610,0.3302
Decision Tree,0.8143,0.7466,0.6611,0.3293,0.4396,0.3730
KNN,0.8087,0.7303,0.6228,0.3421,0.4416,0.3599


In [9]:
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

naive_bayes_model = Pipeline(
    steps=[
        ("preprocessing", clone(preprocessor)),
        ("classifier", GaussianNB())
    ]
)

svm_model = Pipeline(
    steps=[
        ("preprocessing", clone(preprocessor)),
        (
            "classifier",
            SVC(
                kernel="rbf",
                C=1.0,
                gamma="scale",
                probability=False,
                cache_size=512,
                random_state=42
            )
        )
    ]
)

remaining_models = {
    "Naive Bayes": (
        naive_bayes_model,
        "model/naive_bayes.joblib"
    ),
    "Support Vector Machine": (
        svm_model,
        "model/svm.joblib"
    )
}

for name, (model, file_path) in remaining_models.items():
    print(f"Training {name}...", flush=True)
    model.fit(X_train, y_train)
    results[name] = evaluate_model(model, X_test, y_test)
    joblib.dump(model, file_path)

    print(f"Saved {file_path}")
    print("Confusion matrix: rows = actual, columns = predicted; order = [0, 1]")
    print(confusion_matrix(y_test, model.predict(X_test), labels=[0, 1]))
    print()

comparison = pd.DataFrame.from_dict(results, orient="index")
comparison.index.name = "ML Model Name"
comparison.to_csv("model_comparison.csv")

display(comparison.round(4))

print("\nSaved model_comparison.csv")

Training Naive Bayes...
Saved model/naive_bayes.joblib
Confusion matrix: rows = actual, columns = predicted; order = [0, 1]
[[ 453 4220]
 [  38 1289]]

Training Support Vector Machine...


/home/cloud/anaconda3/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Saved model/svm.joblib
Confusion matrix: rows = actual, columns = predicted; order = [0, 1]
[[4447  226]
 [ 876  451]]



,Accuracy,AUC,Precision,Recall,F1,MCC
ML Model Name,,,,,,
Logistic Regression,0.8088,0.7100,0.6923,0.2442,0.3610,0.3302
Decision Tree,0.8143,0.7466,0.6611,0.3293,0.4396,0.3730
KNN,0.8087,0.7303,0.6228,0.3421,0.4416,0.3599
Naive Bayes,0.2903,0.7240,0.2340,0.9714,0.3771,0.1034
Support Vector Machine,0.8163,0.7202,0.6662,0.3399,0.4501,0.3824



Saved model_comparison.csv


In [11]:
%pip install streamlit matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 118.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.2/797.2 kB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.0/55.0 MB 155.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 137.5 MB/s eta 0:00:00
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.11.0
    Uninstalling typing_extensions-4.11.0:
      Successfully uninstalled typing_extensions-4.11.0
  Attempting uninstall: protobuf
    Found existing installation: protobuf 4.25.3
    Uninstalling protobuf-4.25.3:
      Successfully uninstalled protobuf-4.25.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.17.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.20.3, but you have protobuf 7.

In [13]:
from importlib.metadata import version
from pathlib import Path

packages = [
    "streamlit",
    "scikit-learn",
    "numpy",
    "pandas",
    "matplotlib",
    "joblib"
]

requirements = "\n".join(
    f"{package}=={version(package)}"
    for package in packages
)

Path("requirements.txt").write_text(
    requirements + "\n",
    encoding="utf-8"
)

print(Path("requirements.txt").read_text())

streamlit==1.67.0
scikit-learn==1.9.1
numpy==2.5.4
pandas==2.2.3
matplotlib==3.10.1
joblib==1.4.2



In [15]:
%%writefile app.py
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import streamlit as st

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    ConfusionMatrixDisplay,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score
)

st.set_page_config(
    page_title="Credit Default Model Explorer",
    page_icon="💳",
    layout="wide"
)

BASE_DIR = Path(__file__).resolve().parent

MODEL_FILES = {
    "Logistic Regression": "logistic_regression.joblib",
    "Decision Tree": "decision_tree.joblib",
    "KNN": "knn.joblib",
    "Naive Bayes": "naive_bayes.joblib",
    "Support Vector Machine": "svm.joblib"
}

FEATURES = [f"X{i}" for i in range(1, 24)]
TARGET = "default"

@st.cache_resource
def load_model(filename):
    return joblib.load(BASE_DIR / "model" / filename)

st.title("Credit Default Model Explorer")
st.write(
    "Compare five classifiers trained on the UCI Default of Credit Card "
    "Clients dataset."
)
st.caption(
    "Target: 0 = no default; 1 = default. "
    "An educational comparison using a held-out test set."
)

with st.sidebar:
    st.header("Evaluation settings")
    selected_model = st.selectbox(
        "Choose a classifier",
        list(MODEL_FILES)
    )

    sample_path = BASE_DIR / "test_data.csv"

    if sample_path.exists():
        st.download_button(
            "Download held-out test CSV",
            data=sample_path.read_bytes(),
            file_name="test_data.csv",
            mime="text/csv"
        )

    st.write(
        "Upload test data with columns X1 through X23 and default. "
        "Saved preprocessing is applied automatically."
    )

comparison_path = BASE_DIR / "model_comparison.csv"

if comparison_path.exists():
    with st.expander("Compare all models on the original held-out test set"):
        comparison = pd.read_csv(comparison_path, index_col=0)
        st.dataframe(comparison.round(4), use_container_width=True)
        st.caption(
            "These are saved training-run results. "
            "The evaluation below uses your uploaded CSV."
        )

uploaded_file = st.file_uploader(
    "Upload labeled test data",
    type=["csv"]
)

if uploaded_file is None:
    st.info(
        "Download the held-out test CSV from the sidebar, "
        "then upload it here to evaluate a model."
    )
    st.stop()

try:
    data = pd.read_csv(uploaded_file)
except Exception as exc:
    st.error(f"Could not read the CSV: {exc}")
    st.stop()

if data.empty:
    st.error("The uploaded CSV contains no rows.")
    st.stop()

required_columns = FEATURES + [TARGET]
missing_columns = [
    column for column in required_columns
    if column not in data.columns
]

if missing_columns:
    st.error(f"Missing required columns: {', '.join(missing_columns)}")
    st.stop()

try:
    inputs = data[FEATURES].apply(pd.to_numeric, errors="raise")
    target = pd.to_numeric(data[TARGET], errors="raise")
except (ValueError, TypeError):
    st.error("All input features and the default column must be numeric.")
    st.stop()

if not np.isfinite(inputs.to_numpy(dtype=float)).all():
    st.error("Input features must not contain missing or infinite values.")
    st.stop()

if not target.isin([0, 1]).all():
    st.error("The default column must contain only 0 and 1.")
    st.stop()

target = target.astype(int)

st.subheader("Uploaded test data")
left, right = st.columns(2)
left.metric("Test records", f"{len(data):,}")
right.metric("Actual default rate", f"{target.mean():.2%}")

with st.expander("Preview uploaded data"):
    st.dataframe(data.head(20), use_container_width=True)

if not st.button("Evaluate selected model", type="primary"):
    st.stop()

try:
    model = load_model(MODEL_FILES[selected_model])

    with st.spinner(f"Evaluating {selected_model}..."):
        predictions = model.predict(inputs)

        if hasattr(model, "predict_proba"):
            positive_index = list(model.classes_).index(1)
            scores = model.predict_proba(inputs)[:, positive_index]
        else:
            scores = model.decision_function(inputs)
except Exception as exc:
    st.error(f"Model evaluation failed: {exc}")
    st.stop()

auc = (
    roc_auc_score(target, scores)
    if target.nunique() == 2
    else None
)

metrics = {
    "Accuracy": accuracy_score(target, predictions),
    "AUC": auc,
    "Precision": precision_score(target, predictions, zero_division=0),
    "Recall": recall_score(target, predictions, zero_division=0),
    "F1": f1_score(target, predictions, zero_division=0),
    "MCC": matthews_corrcoef(target, predictions)
}

st.subheader(f"{selected_model}: evaluation results")

for row in [
    ["Accuracy", "AUC", "Precision"],
    ["Recall", "F1", "MCC"]
]:
    columns = st.columns(3)

    for column, name in zip(columns, row):
        value = metrics[name]
        column.metric(
            name,
            "N/A" if value is None else f"{value:.4f}"
        )

if auc is None:
    st.warning(
        "AUC requires both target classes. "
        "This CSV contains only one class."
    )

st.caption(
    "Precision, recall, and F1 use default (1) as the positive class. "
    "SVM AUC uses decision scores."
)

left, right = st.columns(2)

with left:
    st.subheader("Confusion matrix")

    matrix = confusion_matrix(target, predictions, labels=[0, 1])
    fig, ax = plt.subplots(figsize=(5, 4))

    ConfusionMatrixDisplay(
        confusion_matrix=matrix,
        display_labels=["No default", "Default"]
    ).plot(ax=ax, cmap="Blues", colorbar=False)

    fig.tight_layout()
    st.pyplot(fig)
    plt.close(fig)

with right:
    st.subheader("Classification report")

    report = classification_report(
        target,
        predictions,
        labels=[0, 1],
        target_names=["No default", "Default"],
        output_dict=True,
        zero_division=0
    )

    st.dataframe(
        pd.DataFrame(report).transpose().round(4),
        use_container_width=True
    )

prediction_data = data[required_columns].copy()
prediction_data["predicted_default"] = predictions

st.download_button(
    "Download predictions",
    data=prediction_data.to_csv(index=False).encode("utf-8"),
    file_name="credit_default_predictions.csv",
    mime="text/csv"
)

Writing app.py


In [17]:
from pathlib import Path

required_files = [
    "app.py",
    "requirements.txt",
    "test_data.csv",
    "model_comparison.csv",
    "model/logistic_regression.joblib",
    "model/decision_tree.joblib",
    "model/knn.joblib",
    "model/naive_bayes.joblib",
    "model/svm.joblib"
]

for filename in required_files:
    path = Path(filename)
    status = "OK" if path.is_file() else "MISSING"
    print(f"{status}: {filename}")

OK: app.py
OK: requirements.txt
OK: test_data.csv
OK: model_comparison.csv
OK: model/logistic_regression.joblib
OK: model/decision_tree.joblib
OK: model/knn.joblib
OK: model/naive_bayes.joblib
OK: model/svm.joblib


In [19]:
from pathlib import Path
import pandas as pd

comparison = pd.read_csv("model_comparison.csv", index_col=0)

header = "| ML Model Name | " + " | ".join(comparison.columns) + " |"
separator = "| " + " | ".join(["---"] * (len(comparison.columns) + 1)) + " |"

rows = [
    "| " + str(name) + " | "
    + " | ".join(f"{value:.4f}" for value in row)
    + " |"
    for name, row in comparison.iterrows()
]

metrics_table = "\n".join([header, separator] + rows)

readme = f"""
Credit Default Model Explorer
============================

Problem statement
-----------------

Compare five machine-learning classifiers for predicting whether a credit
card client will default on payment next month. Provide an interactive
Streamlit application for evaluating saved models on labeled test data.

Dataset description
-------------------

Source: UCI Machine Learning Repository — Default of Credit Card Clients

Dataset link:
https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients

The dataset contains 30,000 records and 23 input features covering credit
limit, demographic information, repayment history, bill amounts, and
previous payments.

The customer ID is excluded from training.

Target: default
- 0: No default
- 1: Default

Class distribution:
- No default: 23,364 records
- Default: 6,636 records

A stratified 80/20 split produces 24,000 training records and 6,000 test
records, using random_state=42.

X2, X3, and X4 are one-hot encoded. Remaining inputs are standardized.
Preprocessing is fitted only on training data and saved with each model
as a scikit-learn pipeline.

GitHub repository link
---------------------

REPLACE_WITH_YOUR_GITHUB_REPOSITORY_URL

Live Streamlit application
-------------------------

REPLACE_WITH_YOUR_STREAMLIT_APP_URL

Models used
-----------

- Logistic Regression
- Decision Tree Classifier
- K-Nearest Neighbors Classifier
- Gaussian Naive Bayes
- Support Vector Machine with an RBF kernel

Evaluation results
------------------

All models are evaluated on the same held-out test set.
Precision, recall, and F1 use default (1) as the positive class.
AUC uses positive-class probabilities, except for SVM, which uses
decision scores.

{metrics_table}

Observations on model performance
---------------------------------

Add your own interpretation for each model below. Use the comparison
table and confusion matrices from your notebook.

| ML Model Name | Observation about model performance |
| --- | --- |
| Logistic Regression | TO COMPLETE |
| Decision Tree | TO COMPLETE |
| KNN | TO COMPLETE |
| Naive Bayes | TO COMPLETE |
| Support Vector Machine | TO COMPLETE |
| Overall winner for this dataset | TO COMPLETE |

Application features
--------------------

- Upload labeled test data in CSV format.
- Select one of five saved classifiers.
- Display Accuracy, AUC, Precision, Recall, F1, and MCC.
- Display a confusion matrix and classification report.
- Compare saved results from the original held-out test set.
- Download test data and model predictions.
- Evaluate uploaded data without retraining.

Uploaded CSV files must include X1 through X23 and the default column.

Project files
-------------

- app.py: Streamlit application.
- requirements.txt: Runtime dependencies.
- credit_default_training.ipynb: Training and evaluation notebook.
- test_data.csv: Held-out test features and labels.
- model_comparison.csv: Evaluation results for all five models.
- model/: Five saved model pipelines.

Running the application
-----------------------

Install dependencies:

    pip install -r requirements.txt

Start the application:

    streamlit run app.py

Limitations
-----------

The dataset is imbalanced. Accuracy alone is insufficient for comparing
default detection. Results are from one held-out split and do not
establish suitability for real-world lending decisions.
"""

Path("README.md").write_text(readme.strip() + "\n", encoding="utf-8")

print("Created README.md")

Created README.md
